# 서울형 키즈카페 포털 + 네이버 검색 기반 데이터 수집 코드

- `서울형 키즈카페 포털`: 공식 매장 메타데이터
- `네이버 검색 API`: 블로그 / 카페글 검색 결과

출력
- `outputs/raw/places.csv`: 매장 기준 테이블
- `outputs/raw/review_docs.csv`: 후기성 문서 기준 테이블
- `outputs/raw/place_features.csv`: 추천 필터용 파생 feature 테이블

작업 흐름
1. 상단 `사용자 설정` 셀에서 API와 수집 범위를 정합니다.
2. 서울형 키즈카페 포털의 목록 페이지를 순회합니다.
3. 각 매장의 상세/오시는길 페이지를 추가로 읽어 메타데이터를 보강합니다.
4. 네이버 블로그/카페글 검색 API로 리뷰성 문서를 수집합니다.
5. `places`, `review_docs`, `place_features` 형태로 나눠서 저장합니다.


In [ ]:
# =============================
# 사용자 설정
# =============================
# 이 셀만 먼저 수정하면 다른 사람도 쉽게 실행할 수 있도록 설정을 위쪽에 모았습니다.
# 직접 값을 넣어도 되고, 비워둔 뒤 `.env.local`을 사용할 수도 있습니다.

# 1. API 설정
USE_ENV_FILE = True              # True면 .env.local 파일도 함께 읽습니다.
ENV_FILE_NAME = '.env.local'     # API 키를 저장한 환경변수 파일 이름입니다.
NAVER_CLIENT_ID = ''             # 직접 입력하려면 여기에 값 입력, 비워두면 .env.local 사용
NAVER_CLIENT_SECRET = ''         # 직접 입력하려면 여기에 값 입력, 비워두면 .env.local 사용

# 2. 포털 수집 범위 설정
FACILITY_STYLE_CODE = '2001'     # 2001=일반형, 2002=여기저기형
DISTRICT_CODES_TO_USE = None     # None이면 전체 자치구, 예: ['11680'] 은 강남구만 수집
MAX_PAGES = None                 # None이면 끝까지 수집, 테스트 시에는 1~2 추천
ENRICH_DETAIL = True             # 이용안내 상세 페이지까지 수집할지 여부
ENRICH_DIRECTIONS = True         # 오시는길 상세 페이지까지 수집할지 여부
ENRICH_PROGRAM = True            # 체험안내 페이지의 프로그램 설명까지 수집할지 여부

# 3. 네이버 리뷰성 문서 수집 설정
INCLUDE_BLOG = True
BLOG_DOCS_PER_PLACE = 5
INCLUDE_CAFE = True
CAFE_DOCS_PER_PLACE = 5
REVIEW_TERMS = ['후기', '리뷰', '추천', '좋았', '별로', '개인적']
DOC_SORT = 'sim'                # date=최신순, sim=정확도순

# 4. 리뷰 품질 설정
FILTER_PROMOTIONAL_DOCS = True
PROMOTIONAL_KEYWORDS = [
    '광고', '유료광고', '협찬', '체험단', '원고료', '소정의 원고료',
    '소정의', '지원받아', '제공받아', '방문권', '초대권', '파트너스', '서포터즈'
]
INCLUDE_REVIEW_DERIVED_FEATURES = True   # 리뷰 snippet 기반 feature를 place_features에 추가할지 여부

# 5. 요청 안정성 설정
REQUEST_INTERVAL_SECONDS = 0.45  # 너무 낮추면 429 속도 제한에 걸릴 수 있습니다.
REQUEST_MAX_RETRIES = 4

# 6. 저장 설정
# 결과 파일은 항상 outputs/raw/places.csv, outputs/raw/review_docs.csv, outputs/raw/place_features.csv 로 저장됩니다.


In [ ]:
# 노트북을 처음 실행하는 환경이라면 필요한 패키지를 설치합니다.
# 이미 설치돼 있다면 이 셀은 건너뛰셔도 됩니다.

%pip install -q pandas requests beautifulsoup4 lxml


In [ ]:
from __future__ import annotations

import html
import os
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
from urllib.parse import parse_qs, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup, Tag


# 현재 노트북이 있는 폴더를 프로젝트 루트로 가정합니다.
PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / ENV_FILE_NAME
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RAW_OUTPUT_DIR = OUTPUT_DIR / "raw"
OUTPUT_DIR.mkdir(exist_ok=True)
RAW_OUTPUT_DIR.mkdir(exist_ok=True)


# 서울형 키즈카페 포털 관련 기본 URL입니다.
PORTAL_BASE_URL = "https://umppa.seoul.go.kr/icare/user/kidsCafe/"
PORTAL_LIST_URL = urljoin(PORTAL_BASE_URL, "BD_selectKidsCafeList.do")

# 네이버 검색 API 기본 URL입니다.
NAVER_SEARCH_BASE_URL = "https://openapi.naver.com/v1/search"

# 서울시 자치구 코드 목록입니다.
DISTRICT_CODES = [
    "1000",
    "11680", "11740", "11305", "11500", "11620", "11215", "11530",
    "11545", "11350", "11320", "11230", "11590", "11440", "11410",
    "11650", "11200", "11290", "11710", "11470", "11560", "11170",
    "11380", "11110", "11140", "11260",
]

FACILITY_STYLE_MAP = {
    "2001": "일반형",
    "2002": "여기저기형",
}

DEFAULT_FCLTY_STYLE = "2001"
DEFAULT_ROW_PER_PAGE = 5
DEFAULT_REVIEW_TERMS = ["후기", "리뷰", "추천"]
DEFAULT_DOC_SORT = "date"
DEFAULT_REQUEST_INTERVAL_SECONDS = REQUEST_INTERVAL_SECONDS
MAX_RETRY_COUNT = REQUEST_MAX_RETRIES

HTML_TAG_RE = re.compile(r"<[^>]+>")
SPACE_RE = re.compile(r"\s+")
AGE_RANGE_RE = re.compile(r"(\d+)\s*~\s*(\d+)\s*세")
CAPACITY_RE = re.compile(r"(개인|단체)\s*(\d+)\s*명")
PHONE_RE = re.compile(r"\b\d{2,4}-\d{3,4}-\d{4}\b")

session = requests.Session()
session.headers.update({"User-Agent": "seoul-kidscafe-portal-notebook/1.0"})


In [ ]:
@dataclass
class PortalPlace:
    # 서울형 키즈카페 포털에서 수집한 매장 메타데이터를 담는 자료형입니다.
    place_id: str
    place_name: str
    facility_style_code: str = ""
    facility_type: str = ""
    image_url: str = ""
    address: str = ""
    phone: str = ""
    capacity_personal: str = ""
    capacity_group: str = ""
    age_text: str = ""
    age_min: str = ""
    age_max: str = ""
    operating_day_text: str = ""
    closed_day_text: str = ""
    operating_hours_text: str = ""
    price_info_text: str = ""
    usage_age_detail_text: str = ""
    usage_rules_text: str = ""
    program_info_text: str = ""
    subway_info: str = ""
    bus_info: str = ""
    car_info: str = ""
    parking_info: str = ""
    detail_url: str = ""
    notice_url: str = ""
    program_url: str = ""
    directions_url: str = ""
    reservation_url: str = ""


@dataclass
class ReviewDoc:
    # 네이버 블로그/카페글 검색 결과를 담는 자료형입니다.
    place_id: str
    place_name: str
    source_type: str
    search_query: str
    title: str
    doc_url: str
    content: str
    author_name: str = ""
    author_link: str = ""
    post_date: str = ""


def load_local_env_file(env_path: Path) -> None:
    # .env.local 파일이 있으면 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 등을 로드합니다.
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export ") :].strip()
        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value.startswith(("'", '"')) and value.endswith(("'", '"')):
            value = value[1:-1]

        os.environ.setdefault(key, value)


def normalize_space(value: str) -> str:
    return SPACE_RE.sub(" ", value.replace("\xa0", " ").strip())


def normalize_text(value: str) -> str:
    return normalize_space(value).lower()


def strip_tags(value: str) -> str:
    return html.unescape(HTML_TAG_RE.sub("", value or "")).strip()


def clean_label(value: str) -> str:
    # 포털의 "주     소" 같이 공백이 섞인 라벨을 "주소"처럼 정리합니다.
    return re.sub(r"\s+", "", value.replace("\xa0", " "))


def absolute_url(url: str) -> str:
    if not url:
        return ""
    return urljoin(PORTAL_BASE_URL, url)


def extract_query_param(url: str, key: str) -> str:
    if not url:
        return ""
    parsed = urlparse(url)
    values = parse_qs(parsed.query).get(key, [])
    return values[0] if values else ""


def extract_age_range(age_text: str) -> tuple[str, str]:
    match = AGE_RANGE_RE.search(age_text or "")
    if not match:
        return "", ""
    return match.group(1), match.group(2)


def extract_capacity_values(capacity_text: str) -> tuple[str, str]:
    personal = ""
    group = ""
    for label, value in CAPACITY_RE.findall(capacity_text or ""):
        if label == "개인":
            personal = value
        elif label == "단체":
            group = value
    return personal, group


def extract_phone(value: str) -> str:
    match = PHONE_RE.search(value or "")
    return match.group(0) if match else ""


def guess_parking_info(car_info: str) -> str:
    if "주차" in (car_info or ""):
        return normalize_space(car_info)
    return ""


def build_request_state() -> dict[str, Any]:
    return {"last_request_ts": 0.0, "total_wait_seconds": 0}


def request_text(
    url: str,
    *,
    params: list[tuple[str, str]] | dict[str, Any] | None = None,
    headers: dict[str, str] | None = None,
    request_state: dict[str, Any] | None = None,
) -> str:
    # 공통 GET 요청 함수입니다. 너무 빠르게 때리지 않도록 간격을 둡니다.
    request_state = request_state or build_request_state()
    last_request_ts = float(request_state.get("last_request_ts", 0.0) or 0.0)
    elapsed = time.monotonic() - last_request_ts if last_request_ts else None

    if elapsed is not None and elapsed < DEFAULT_REQUEST_INTERVAL_SECONDS:
        time.sleep(DEFAULT_REQUEST_INTERVAL_SECONDS - elapsed)

    for attempt in range(1, MAX_RETRY_COUNT + 1):
        response = session.get(url, params=params, headers=headers, timeout=20)
        request_state["last_request_ts"] = time.monotonic()

        if response.status_code != 429:
            response.raise_for_status()
            return response.text

        wait_seconds = min(20, 3 * attempt)
        request_state["total_wait_seconds"] = int(request_state.get("total_wait_seconds", 0)) + wait_seconds
        if attempt >= MAX_RETRY_COUNT:
            response.raise_for_status()
        time.sleep(wait_seconds)

    raise RuntimeError("Unexpected request retry flow")


def request_json(
    url: str,
    *,
    params: dict[str, Any],
    headers: dict[str, str],
    request_state: dict[str, Any],
) -> dict[str, Any]:
    last_request_ts = float(request_state.get("last_request_ts", 0.0) or 0.0)
    elapsed = time.monotonic() - last_request_ts if last_request_ts else None

    if elapsed is not None and elapsed < DEFAULT_REQUEST_INTERVAL_SECONDS:
        time.sleep(DEFAULT_REQUEST_INTERVAL_SECONDS - elapsed)

    for attempt in range(1, MAX_RETRY_COUNT + 1):
        response = session.get(url, params=params, headers=headers, timeout=20)
        request_state["last_request_ts"] = time.monotonic()

        if response.status_code != 429:
            response.raise_for_status()
            return response.json()

        wait_seconds = min(20, 3 * attempt)
        request_state["total_wait_seconds"] = int(request_state.get("total_wait_seconds", 0)) + wait_seconds
        if attempt >= MAX_RETRY_COUNT:
            response.raise_for_status()
        time.sleep(wait_seconds)

    raise RuntimeError("Unexpected request retry flow")


if USE_ENV_FILE:
    load_local_env_file(ENV_PATH)

# 직접 입력한 값이 있으면 그것을 우선 사용하고, 비어 있으면 .env.local / 환경변수 값을 사용합니다.
NAVER_CLIENT_ID = NAVER_CLIENT_ID or os.getenv("NAVER_CLIENT_ID", "")
NAVER_CLIENT_SECRET = NAVER_CLIENT_SECRET or os.getenv("NAVER_CLIENT_SECRET", "")

# 자치구 코드를 따로 지정하지 않으면 전체 자치구를 사용합니다.
SELECTED_DISTRICT_CODES = DISTRICT_CODES_TO_USE or DISTRICT_CODES


In [ ]:
def build_portal_list_params(
    *,
    page: int,
    row_per_page: int = DEFAULT_ROW_PER_PAGE,
    facility_style_code: str = DEFAULT_FCLTY_STYLE,
    district_codes: list[str] | None = None,
    use_age: str = "",
    search_value: str = "",
) -> list[tuple[str, str]]:
    # 포털 목록 페이지용 query string 파라미터를 구성합니다.
    district_codes = district_codes or DISTRICT_CODES

    params: list[tuple[str, str]] = [
        ("q_hiddenVal", "1"),
        ("q_fcltyId", ""),
        ("q_rowPerPage", str(row_per_page)),
        ("q_currPage", str(page)),
        ("q_sortName", ""),
        ("q_sortOrder", ""),
        ("q_fcltyStle", facility_style_code),
        ("q_searchVal", search_value),
    ]

    for code in district_codes:
        params.append(("q_atdrcCode", code))

    params.append(("q_useAge", use_age))
    return params


def parse_portal_list_page(html_text: str, *, facility_style_code: str) -> list[PortalPlace]:
    # 목록 페이지의 카드형 블록(div.kidscafe_wrap)을 파싱해서 매장 기본 정보를 추출합니다.
    soup = BeautifulSoup(html_text, "lxml")
    places: list[PortalPlace] = []

    for wrap in soup.select("div.board_kidscafe div.kidscafe_wrap"):
        title_tag = wrap.select_one("h5")
        place_name = normalize_space(title_tag.get_text(" ", strip=True)) if title_tag else ""
        if not place_name:
            continue

        label_value_map: dict[str, str] = {}
        dl = wrap.select_one("div.kidscafe_info dl")
        if dl:
            dts = dl.find_all("dt", recursive=False)
            dds = dl.find_all("dd", recursive=False)
            for dt, dd in zip(dts, dds):
                label = clean_label(dt.get_text(" ", strip=True))
                value = normalize_space(dd.get_text(" ", strip=True))
                label_value_map[label] = value

        capacity_text = label_value_map.get("이용정원", "")
        age_text = label_value_map.get("이용연령", "")
        age_min, age_max = extract_age_range(age_text)
        capacity_personal, capacity_group = extract_capacity_values(capacity_text)

        links: dict[str, str] = {}
        for anchor in wrap.select("div.btn_wrap a[href]"):
            link_text = normalize_space(anchor.get_text(" ", strip=True))
            links[link_text] = absolute_url(anchor.get("href"))

        detail_url = links.get("이용안내", "")
        place_id = extract_query_param(detail_url, "q_fcltyId")
        if not place_id:
            place_id = extract_query_param(links.get("예약 신청", ""), "q_fcltyId")

        image_tag = wrap.select_one("div.kidscafe_image img")
        image_url = absolute_url(image_tag.get("src")) if image_tag else ""

        place = PortalPlace(
            place_id=place_id or normalize_text(place_name),
            place_name=place_name,
            facility_style_code=facility_style_code,
            facility_type=FACILITY_STYLE_MAP.get(facility_style_code, facility_style_code),
            image_url=image_url,
            address=label_value_map.get("주소", ""),
            phone=extract_phone(label_value_map.get("전화번호", "")),
            capacity_personal=capacity_personal,
            capacity_group=capacity_group,
            age_text=age_text,
            age_min=age_min,
            age_max=age_max,
            detail_url=detail_url,
            notice_url=links.get("공지사항", ""),
            program_url=links.get("체험안내", ""),
            directions_url=links.get("오시는길", ""),
            reservation_url=links.get("예약 신청", ""),
        )
        places.append(place)

    return places


def parse_detail_sections(center_info: Tag | None) -> dict[str, str]:
    # 상세 페이지의 center_info 영역에서 h3 제목 단위로 본문을 묶어서 반환합니다.
    if center_info is None:
        return {}

    sections: dict[str, str] = {}
    for heading in center_info.select("h3.sub_title03"):
        title = normalize_space(heading.get_text(" ", strip=True))
        parts: list[str] = []

        for sibling in heading.next_siblings:
            if isinstance(sibling, Tag):
                if sibling.name == "h3":
                    break
                text = normalize_space(sibling.get_text("\n", strip=True))
                if text:
                    parts.append(text)

        if parts:
            sections[title] = "\n\n".join(parts)

    return sections


def enrich_place_from_detail(place: PortalPlace, request_state: dict[str, Any]) -> PortalPlace:
    # 이용안내 상세 페이지에서 운영일, 휴관일, 운영시간, 이용요금 등을 보강합니다.
    if not place.detail_url:
        return place

    html_text = request_text(place.detail_url, request_state=request_state)
    soup = BeautifulSoup(html_text, "lxml")

    summary_map: dict[str, str] = {}
    for li in soup.select("div.bx-right li"):
        label_tag = li.find("b")
        if label_tag is None:
            continue
        label = clean_label(label_tag.get_text(" ", strip=True))
        value = normalize_space(li.get_text(" ", strip=True).replace(label_tag.get_text(" ", strip=True), "", 1))
        summary_map[label] = value

    sections = parse_detail_sections(soup.select_one("div.center_info"))

    place.address = place.address or summary_map.get("주소", "")
    place.phone = place.phone or extract_phone(summary_map.get("연락처", ""))
    place.operating_day_text = summary_map.get("운영일", "")
    place.closed_day_text = summary_map.get("휴관일", "")
    place.operating_hours_text = sections.get("운영시간", "")
    place.price_info_text = sections.get("이용료", "") or sections.get("이용요금", "")
    place.usage_age_detail_text = sections.get("이용연령", "")

    usage_rule_parts = []
    for key in ["이용규칙", "이용수칙", "유의사항", "주의사항", "이용안내"]:
        if sections.get(key):
            usage_rule_parts.append(sections[key])
    place.usage_rules_text = "\n\n".join(usage_rule_parts)

    return place


def enrich_place_from_directions(place: PortalPlace, request_state: dict[str, Any]) -> PortalPlace:
    # 오시는길 페이지에서 지하철/버스/자가용 정보를 보강합니다.
    if not place.directions_url:
        return place

    html_text = request_text(place.directions_url, request_state=request_state)
    soup = BeautifulSoup(html_text, "lxml")

    row_map: dict[str, str] = {}
    for tr in soup.select("div.center_info table tbody tr"):
        ths = tr.find_all("th")
        tds = tr.find_all("td")
        if len(ths) == len(tds):
            for th, td in zip(ths, tds):
                label = clean_label(th.get_text(" ", strip=True))
                value = normalize_space(td.get_text("\n", strip=True))
                row_map[label] = value

    place.address = place.address or row_map.get("주소", "")
    place.phone = place.phone or extract_phone(row_map.get("연락처", ""))
    place.subway_info = row_map.get("지하철이용시", "")
    place.bus_info = row_map.get("버스이용시", "")
    place.car_info = row_map.get("자가용이용시", "")
    place.parking_info = guess_parking_info(place.car_info)

    return place


def enrich_place_from_program(place: PortalPlace, request_state: dict[str, Any]) -> PortalPlace:
    # 체험안내 페이지에서 놀이시설/프로그램 설명 텍스트를 보강합니다.
    if not place.program_url:
        return place

    html_text = request_text(place.program_url, request_state=request_state)
    soup = BeautifulSoup(html_text, "lxml")

    program_items: list[str] = []
    for li in soup.select("div.kidscafe_details li"):
        title_tag = li.select_one("h5")
        desc_tag = li.select_one("div.kidscafe_details_info div")
        title = normalize_space(title_tag.get_text(" ", strip=True)) if title_tag else ""
        description = normalize_space(desc_tag.get_text(" ", strip=True)) if desc_tag else ""

        if title and description:
            program_items.append(f"{title}: {description}")
        elif title or description:
            program_items.append(title or description)

    if program_items:
        place.program_info_text = "\n".join(program_items)

    return place


def collect_portal_places(
    *,
    facility_style_code: str = DEFAULT_FCLTY_STYLE,
    district_codes: list[str] | None = None,
    max_pages: int | None = None,
    enrich_detail: bool = True,
    enrich_directions: bool = True,
    enrich_program: bool = True,
) -> list[PortalPlace]:
    # 포털 목록을 끝까지 순회하면서 매장 메타데이터를 수집합니다.
    request_state = build_request_state()
    all_places: list[PortalPlace] = []
    seen_ids: set[str] = set()
    page = 1

    while True:
        params = build_portal_list_params(
            page=page,
            facility_style_code=facility_style_code,
            district_codes=district_codes,
        )
        html_text = request_text(PORTAL_LIST_URL, params=params, request_state=request_state)
        page_places = parse_portal_list_page(html_text, facility_style_code=facility_style_code)

        if not page_places:
            break

        for place in page_places:
            if place.place_id in seen_ids:
                continue
            seen_ids.add(place.place_id)
            all_places.append(place)

        print(f"[portal] page={page} collected={len(page_places)} cumulative={len(all_places)}")

        page += 1
        if max_pages is not None and page > max_pages:
            break

    if enrich_detail:
        for index, place in enumerate(all_places, start=1):
            enrich_place_from_detail(place, request_state)
            print(f"[detail] {index}/{len(all_places)} {place.place_name}")

    if enrich_directions:
        for index, place in enumerate(all_places, start=1):
            enrich_place_from_directions(place, request_state)
            print(f"[directions] {index}/{len(all_places)} {place.place_name}")

    if enrich_program:
        for index, place in enumerate(all_places, start=1):
            enrich_place_from_program(place, request_state)
            print(f"[program] {index}/{len(all_places)} {place.place_name}")

    waited = int(request_state.get("total_wait_seconds", 0))
    if waited:
        print(f"[portal] total_wait_seconds={waited}")

    return all_places


def portal_places_to_dataframe(places: list[PortalPlace]) -> pd.DataFrame:
    return pd.DataFrame([asdict(place) for place in places])


In [ ]:
def build_naver_headers(client_id: str, client_secret: str) -> dict[str, str]:
    return {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret,
        "User-Agent": "seoul-kidscafe-portal-notebook/1.0",
    }


def extract_district(address: str) -> str:
    match = re.search(r"서울(?:특별시)?\s+(\S+구)", address or "")
    return match.group(1) if match else ""


def build_review_queries(place: PortalPlace, review_terms: list[str]) -> list[str]:
    # 정확도를 높이기 위해 매장명 기반으로 여러 검색어를 만듭니다.
    district = extract_district(place.address)
    queries = [place.place_name]

    if district:
        queries.append(f"{place.place_name} {district}")

    for term in review_terms:
        queries.append(f"{place.place_name} {term}")
        if district:
            queries.append(f"{place.place_name} {district} {term}")

    unique_queries: list[str] = []
    seen: set[str] = set()
    for query in queries:
        if query not in seen:
            seen.add(query)
            unique_queries.append(query)

    return unique_queries


def is_promotional_doc(*, title: str, content: str, keywords: list[str]) -> bool:
    # 광고/협찬/체험단 성격이 강한 문서는 추천 근거 품질을 위해 제외합니다.
    combined_text = normalize_text(f"{title} {content}")
    return any(keyword in combined_text for keyword in keywords)


def search_review_docs(
    *,
    places: list[PortalPlace],
    endpoint: str,
    source_type: str,
    client_id: str,
    client_secret: str,
    docs_per_place: int,
    review_terms: list[str] | None = None,
    sort: str = DEFAULT_DOC_SORT,
) -> list[ReviewDoc]:
    # 네이버 블로그 또는 카페글 검색 API로 리뷰성 문서를 수집합니다.
    review_terms = review_terms or DEFAULT_REVIEW_TERMS
    headers = build_naver_headers(client_id, client_secret)
    request_state = build_request_state()
    results: list[ReviewDoc] = []
    seen: set[tuple[str, str, str]] = set()
    skipped_promotional_total = 0

    for place in places:
        collected_for_place = 0
        skipped_promotional_for_place = 0
        queries = build_review_queries(place, review_terms)

        for query in queries:
            if collected_for_place >= docs_per_place:
                break

            payload = request_json(
                f"{NAVER_SEARCH_BASE_URL}/{endpoint}.json",
                params={
                    "query": query,
                    "display": min(20, docs_per_place * 3),
                    "start": 1,
                    "sort": sort,
                },
                headers=headers,
                request_state=request_state,
            )

            for item in payload.get("items", []):
                if collected_for_place >= docs_per_place:
                    break

                if source_type == "naver_blog":
                    author_name = (item.get("bloggername") or "").strip()
                    author_link = (item.get("bloggerlink") or "").strip()
                    post_date = (item.get("postdate") or "").strip()
                else:
                    author_name = (item.get("cafename") or "").strip()
                    author_link = (item.get("cafeurl") or "").strip()
                    post_date = ""

                title = strip_tags(item.get("title", ""))
                doc_url = (item.get("link") or "").strip()
                content = strip_tags(item.get("description", ""))

                if not title or not doc_url:
                    continue

                if FILTER_PROMOTIONAL_DOCS and is_promotional_doc(
                    title=title,
                    content=content,
                    keywords=PROMOTIONAL_KEYWORDS,
                ):
                    skipped_promotional_for_place += 1
                    skipped_promotional_total += 1
                    continue

                doc = ReviewDoc(
                    place_id=place.place_id,
                    place_name=place.place_name,
                    source_type=source_type,
                    search_query=query,
                    title=title,
                    doc_url=doc_url,
                    content=content,
                    author_name=author_name,
                    author_link=author_link,
                    post_date=post_date,
                )

                dedupe_key = (doc.source_type, doc.place_id, normalize_text(doc.doc_url))
                if dedupe_key in seen:
                    continue

                seen.add(dedupe_key)
                results.append(doc)
                collected_for_place += 1

        print(
            f"[{source_type}] {place.place_name} -> {collected_for_place} docs"
            f" (skipped_promotional={skipped_promotional_for_place})"
        )

    waited = int(request_state.get("total_wait_seconds", 0))
    if waited:
        print(f"[{source_type}] total_wait_seconds={waited}")
    if skipped_promotional_total:
        print(f"[{source_type}] skipped_promotional_total={skipped_promotional_total}")

    return results


def to_int_or_none(value: str | int | None) -> int | None:
    if value is None or value == "":
        return None
    match = re.search(r"\d+", str(value))
    return int(match.group(0)) if match else None


def add_feature_row(
    rows: list[dict[str, Any]],
    *,
    place_id: str,
    feature_name: str,
    feature_value: str,
    source_type: str,
    evidence_text: str,
    confidence: float,
) -> None:
    if feature_value in ("", None):
        return

    rows.append(
        {
            "place_id": place_id,
            "feature_name": feature_name,
            "feature_value": str(feature_value),
            "source_type": source_type,
            "evidence_text": normalize_space(evidence_text or ""),
            "confidence": round(float(confidence), 2),
        }
    )


SELECTED_FEATURE_NAMES = [
    "district",
    "parking_available",
    "parking_paid",
    "reservation_available",
    "toddler_friendly",
    "preschool_friendly",
    "lower_elementary_friendly",
    "weekend_operation",
    "price_level",
    "discount_available",
    "care_service_available",
    "guardian_required",
    "parking_difficulty",
    "group_visit_available",
    "program_role_play",
    "program_media_play",
    "safety_negative",
    "food_allowed",
    "program_active_play",
    "program_creative_play",
    "cleanliness_positive",
    "spacious_positive",
    "crowded_warning",
]


def contains_weekend_reference(text: str) -> bool:
    return bool(re.search(r"(토요일|일요일|주말|매일)", text or ""))


def contains_any_keyword(text: str, keywords: list[str]) -> bool:
    normalized_text = normalize_text(text or "")
    return any(normalize_text(keyword) in normalized_text for keyword in keywords)


def extract_currency_amounts(text: str) -> list[int]:
    amounts: list[int] = []
    for raw_amount in re.findall(r"(\d[\d,]*)\s*원", text or ""):
        try:
            amounts.append(int(raw_amount.replace(",", "")))
        except ValueError:
            continue
    return amounts


def infer_price_level(price_text: str) -> str:
    normalized_text = normalize_text(price_text or "")
    if not normalized_text:
        return ""

    amounts = extract_currency_amounts(price_text)
    if "무료" in normalized_text and not amounts:
        return "free"
    if amounts:
        return "low" if max(amounts) <= 5000 else "paid"
    if any(keyword in normalized_text for keyword in ["유료", "이용료", "입장료", "요금"]):
        return "paid"
    if "무료" in normalized_text:
        return "free"
    return ""


def infer_parking_available(parking_text: str) -> str:
    normalized_text = normalize_text(parking_text or "")
    if not normalized_text:
        return ""
    if any(keyword in normalized_text for keyword in ["주차불가", "주차 불가", "주차없음", "주차 없음", "주차미제공", "주차 미제공"]):
        return "no"
    if "주차" in normalized_text:
        return "yes"
    return ""


def infer_parking_paid(parking_text: str) -> str:
    normalized_text = normalize_text(parking_text or "")
    if not normalized_text or "주차" not in normalized_text:
        return ""
    if any(keyword in normalized_text for keyword in ["무료주차", "무료 주차", "주차비 무료", "주차요금 없음"]):
        return "no"
    if any(keyword in normalized_text for keyword in ["유료", "주차요금", "요금", "정산", "시간당", "30분당", "10분당", "일최대", "일 최대", "최초"]):
        return "yes"
    return ""


def infer_parking_difficulty(parking_text: str) -> str:
    normalized_text = normalize_text(parking_text or "")
    if not normalized_text:
        return ""
    if any(keyword in normalized_text for keyword in ["협소", "혼잡", "어려움", "불편", "대중교통 이용 권장", "주차공간 부족", "주차 면수 부족"]):
        return "difficult"
    if any(keyword in normalized_text for keyword in ["인근 주차장", "공영주차장", "외부 주차장", "별도 주차장"]):
        return "limited"
    if "주차" in normalized_text:
        return "easy"
    return ""


def infer_discount_available(price_text: str) -> str:
    if contains_any_keyword(price_text, ["할인", "감면", "면제", "다둥이", "다자녀", "국가유공자", "장애인", "수급자", "한부모"]):
        return "yes"
    return ""


def infer_care_service_available(price_text: str, usage_text: str) -> str:
    combined_text = " ".join([price_text or "", usage_text or ""])
    if contains_any_keyword(combined_text, ["놀이돌봄", "돌봄서비스", "돌봄 서비스", "돌봄신청", "돌봄 신청"]):
        return "yes"
    return ""


def infer_guardian_required(usage_text: str, age_text: str) -> str:
    combined_text = normalize_space(" ".join([usage_text or "", age_text or ""]))
    if not combined_text:
        return ""
    patterns = [
        r"보호자.{0,12}(동반|동행|함께|필수|필요|반드시|입장|이용)",
        r"(동반|동행|필수|필요|반드시).{0,12}보호자",
        r"보호자\s*미동반.{0,8}(불가|제한)",
    ]
    return "yes" if any(re.search(pattern, combined_text) for pattern in patterns) else ""


def infer_food_allowed(usage_text: str) -> str:
    normalized_text = normalize_text(usage_text or "")
    if not normalized_text:
        return ""
    if any(keyword in normalized_text for keyword in ["음식물 반입 금지", "음식물반입금지", "취식 불가", "취식불가", "물 외 반입 금지", "물외반입금지", "외부 음식 반입 금지"]):
        return "no"
    if any(keyword in normalized_text for keyword in ["음식물 반입 가능", "음식물반입가능", "취식 가능", "취식가능", "간식 가능"]):
        return "yes"
    return ""


def infer_program_feature(program_text: str, keywords: list[str]) -> str:
    return "yes" if contains_any_keyword(program_text, keywords) else ""


def extract_place_feature_rows(place: PortalPlace) -> list[dict[str, Any]]:
    # places 테이블에서 추천용 필터 feature를 추출합니다.
    rows: list[dict[str, Any]] = []
    district = extract_district(place.address)
    age_min = to_int_or_none(place.age_min)
    age_max = to_int_or_none(place.age_max)
    group_capacity = to_int_or_none(place.capacity_group)

    price_text = place.price_info_text or ""
    usage_text = place.usage_rules_text or ""
    age_detail_text = place.usage_age_detail_text or ""
    program_text = place.program_info_text or ""
    parking_text = normalize_space(" ".join([place.parking_info, place.car_info]))
    operating_text = normalize_space(" ".join([place.operating_day_text, place.closed_day_text, place.operating_hours_text]))
    closed_text = place.closed_day_text or ""

    if district:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="district",
            feature_value=district,
            source_type="portal",
            evidence_text=place.address,
            confidence=0.99,
        )

    parking_available = infer_parking_available(parking_text)
    if parking_available:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="parking_available",
            feature_value=parking_available,
            source_type="portal",
            evidence_text=parking_text,
            confidence=0.9,
        )

    parking_paid = infer_parking_paid(parking_text)
    if parking_paid:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="parking_paid",
            feature_value=parking_paid,
            source_type="portal",
            evidence_text=parking_text,
            confidence=0.85,
        )

    parking_difficulty = infer_parking_difficulty(parking_text)
    if parking_difficulty:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="parking_difficulty",
            feature_value=parking_difficulty,
            source_type="portal",
            evidence_text=parking_text,
            confidence=0.75,
        )

    reservation_available = "yes" if place.reservation_url else ""
    if reservation_available:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="reservation_available",
            feature_value=reservation_available,
            source_type="portal",
            evidence_text=place.reservation_url,
            confidence=0.99,
        )

    if group_capacity is not None and group_capacity > 0:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="group_visit_available",
            feature_value="yes",
            source_type="portal",
            evidence_text=f"단체 정원 {group_capacity}명",
            confidence=0.95,
        )

    if age_min is not None or age_max is not None:
        age_evidence = place.age_text or age_detail_text
        if (age_min is None or age_min <= 3) and (age_max is None or age_max >= 2):
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name="toddler_friendly",
                feature_value="yes",
                source_type="portal",
                evidence_text=age_evidence,
                confidence=0.92,
            )
        if (age_min is None or age_min <= 7) and (age_max is None or age_max >= 4):
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name="preschool_friendly",
                feature_value="yes",
                source_type="portal",
                evidence_text=age_evidence,
                confidence=0.92,
            )
        if age_max is not None and age_max >= 8:
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name="lower_elementary_friendly",
                feature_value="yes",
                source_type="portal",
                evidence_text=age_evidence,
                confidence=0.9,
            )

    if operating_text:
        if contains_weekend_reference(closed_text):
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name="weekend_operation",
                feature_value="no",
                source_type="portal",
                evidence_text=operating_text,
                confidence=0.8,
            )
        elif contains_weekend_reference(operating_text):
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name="weekend_operation",
                feature_value="yes",
                source_type="portal",
                evidence_text=operating_text,
                confidence=0.8,
            )

    price_level = infer_price_level(price_text)
    if price_level:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="price_level",
            feature_value=price_level,
            source_type="portal",
            evidence_text=price_text,
            confidence=0.88,
        )

    discount_available = infer_discount_available(price_text)
    if discount_available:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="discount_available",
            feature_value=discount_available,
            source_type="portal",
            evidence_text=price_text,
            confidence=0.9,
        )

    care_service_available = infer_care_service_available(price_text, usage_text)
    if care_service_available:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="care_service_available",
            feature_value=care_service_available,
            source_type="portal",
            evidence_text=normalize_space(" ".join([price_text, usage_text])),
            confidence=0.9,
        )

    guardian_required = infer_guardian_required(usage_text, age_detail_text)
    if guardian_required:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="guardian_required",
            feature_value=guardian_required,
            source_type="portal",
            evidence_text=normalize_space(" ".join([usage_text, age_detail_text])),
            confidence=0.9,
        )

    food_allowed = infer_food_allowed(usage_text)
    if food_allowed:
        add_feature_row(
            rows,
            place_id=place.place_id,
            feature_name="food_allowed",
            feature_value=food_allowed,
            source_type="portal",
            evidence_text=usage_text,
            confidence=0.82,
        )

    program_feature_rules = {
        "program_active_play": ["트램폴린", "클라이밍", "미끄럼틀", "정글짐", "볼풀", "신체놀이", "대근육", "오르기", "점프", "활동"],
        "program_creative_play": ["블록", "만들기", "공예", "색칠", "그리기", "아뜰리에", "퍼즐", "창의", "레고"],
        "program_role_play": ["역할놀이", "역할 놀이", "소꿉", "병원놀이", "요리놀이", "주방놀이", "가게놀이", "직업체험"],
        "program_media_play": ["미디어", "인터랙티브", "디지털", "스크린", "프로젝션", "영상 체험", "체험형 영상"],
    }
    for feature_name, keywords in program_feature_rules.items():
        feature_value = infer_program_feature(program_text, keywords)
        if feature_value:
            add_feature_row(
                rows,
                place_id=place.place_id,
                feature_name=feature_name,
                feature_value=feature_value,
                source_type="portal",
                evidence_text=program_text,
                confidence=0.82,
            )

    return rows


REVIEW_FEATURE_RULES: dict[str, dict[str, Any]] = {
    "spacious_positive": {
        "keywords": ["넓", "쾌적", "넉넉"],
        "group": "space",
        "polarity": "positive",
        "feature_value": "yes",
    },
    "crowded_warning": {
        "keywords": ["붐비", "혼잡", "대기", "사람 많"],
        "group": "space",
        "polarity": "negative",
        "feature_value": "yes",
    },
    "cleanliness_positive": {
        "keywords": ["깨끗", "청결", "깔끔"],
        "group": "cleanliness",
        "polarity": "positive",
        "feature_value": "yes",
    },
    "safety_negative": {
        "keywords": ["위험", "불안", "다치", "낙상", "안전 아쉽"],
        "group": "safety",
        "polarity": "negative",
        "feature_value": "yes",
    },
}


def extract_review_feature_matches(doc: ReviewDoc) -> dict[str, str]:
    # 문서 1건에서 감지된 review feature와 근거 텍스트를 반환합니다.
    combined_text = normalize_space(f"{doc.title} {doc.content}")
    normalized_text = normalize_text(combined_text)
    matches: dict[str, str] = {}

    for feature_name, rule in REVIEW_FEATURE_RULES.items():
        if any(keyword in normalized_text for keyword in rule["keywords"]):
            matches[feature_name] = combined_text

    return matches


def summarize_feature_evidence(evidences: list[str], *, mention_count: int, opposite_count: int) -> str:
    summary = f"mentions={mention_count}; opposite_mentions={opposite_count}"
    sample_text = " || ".join(evidences[:3])
    return f"{summary} | {sample_text}" if sample_text else summary


def calculate_review_feature_confidence(*, mention_count: int, opposite_count: int, source_type_count: int) -> float:
    # 반복 언급이 많을수록 점수를 올리고, 반대 신호가 있으면 점수를 깎습니다.
    base_confidence = 0.45 + min(mention_count, 4) * 0.1 + max(source_type_count - 1, 0) * 0.05
    adjusted_confidence = base_confidence - opposite_count * 0.08
    return max(0.3, min(0.92, adjusted_confidence))


def build_review_feature_rows(docs: list[ReviewDoc]) -> list[dict[str, Any]]:
    # 매장별로 review feature를 집계해 confidence를 계산합니다.
    place_feature_stats: dict[str, dict[str, dict[str, Any]]] = {}
    place_group_counts: dict[str, dict[str, dict[str, int]]] = {}

    for doc in docs:
        matches = extract_review_feature_matches(doc)
        if not matches:
            continue

        place_feature_stats.setdefault(doc.place_id, {})
        place_group_counts.setdefault(doc.place_id, {})

        for feature_name, evidence_text in matches.items():
            rule = REVIEW_FEATURE_RULES[feature_name]
            group_name = rule["group"]
            polarity = rule["polarity"]

            feature_stat = place_feature_stats[doc.place_id].setdefault(
                feature_name,
                {
                    "count": 0,
                    "source_types": set(),
                    "evidences": [],
                },
            )
            feature_stat["count"] += 1
            feature_stat["source_types"].add(doc.source_type)
            if len(feature_stat["evidences"]) < 3:
                feature_stat["evidences"].append(evidence_text)

            group_stat = place_group_counts[doc.place_id].setdefault(
                group_name,
                {"positive": 0, "negative": 0},
            )
            group_stat[polarity] += 1

    rows: list[dict[str, Any]] = []
    for place_id, feature_stats in place_feature_stats.items():
        for feature_name, stat in feature_stats.items():
            rule = REVIEW_FEATURE_RULES[feature_name]
            group_name = rule["group"]
            polarity = rule["polarity"]
            opposite_polarity = "negative" if polarity == "positive" else "positive"
            opposite_count = place_group_counts[place_id][group_name].get(opposite_polarity, 0)
            mention_count = stat["count"]
            net_signal = mention_count - opposite_count

            if net_signal <= 0 and mention_count < 2:
                continue

            confidence = calculate_review_feature_confidence(
                mention_count=mention_count,
                opposite_count=opposite_count,
                source_type_count=len(stat["source_types"]),
            )
            evidence_text = summarize_feature_evidence(
                stat["evidences"],
                mention_count=mention_count,
                opposite_count=opposite_count,
            )

            add_feature_row(
                rows,
                place_id=place_id,
                feature_name=feature_name,
                feature_value=rule["feature_value"],
                source_type="review_aggregate",
                evidence_text=evidence_text,
                confidence=confidence,
            )

    return rows


def normalize_post_date_value(value: str) -> str:
    normalized_text = normalize_space(value)
    if not normalized_text:
        return ""

    digits = re.sub(r"\D", "", normalized_text)
    if len(digits) == 9 and digits.endswith("0"):
        digits = digits[:-1]
    if len(digits) != 8:
        return ""

    return f"{digits[:4]}-{digits[4:6]}-{digits[6:8]}"


def build_places_dataframe(places: list[PortalPlace], *, collected_at: str) -> pd.DataFrame:
    # places 테이블은 매장 1행 기준 테이블입니다.
    rows: list[dict[str, Any]] = []
    for place in places:
        merged_parking_info = normalize_space(" ".join(dict.fromkeys([value for value in [place.parking_info, place.car_info] if value])))
        rows.append(
            {
                "place_id": place.place_id,
                "place_name": place.place_name,
                "facility_type": place.facility_type,
                "district": extract_district(place.address),
                "address": place.address,
                "phone": place.phone,
                "capacity_personal": to_int_or_none(place.capacity_personal),
                "capacity_group": to_int_or_none(place.capacity_group),
                "age_text": place.age_text,
                "age_min": to_int_or_none(place.age_min),
                "age_max": to_int_or_none(place.age_max),
                "operating_day_text": place.operating_day_text,
                "closed_day_text": place.closed_day_text,
                "operating_hours_text": place.operating_hours_text,
                "price_info_text": place.price_info_text,
                "usage_age_detail_text": place.usage_age_detail_text,
                "usage_rules_text": place.usage_rules_text,
                "program_info_text": place.program_info_text,
                "parking_info": merged_parking_info,
                "subway_info": place.subway_info,
                "bus_info": place.bus_info,
                "image_url": place.image_url,
                "detail_url": place.detail_url,
                "notice_url": place.notice_url,
                "directions_url": place.directions_url,
                "reservation_url": place.reservation_url,
                "collected_at": collected_at,
            }
        )

    columns = [
        "place_id", "place_name", "facility_type", "district", "address", "phone",
        "capacity_personal", "capacity_group", "age_text", "age_min", "age_max",
        "operating_day_text", "closed_day_text", "operating_hours_text", "price_info_text",
        "usage_age_detail_text", "usage_rules_text", "program_info_text", "parking_info", "subway_info", "bus_info",
        "image_url", "detail_url", "notice_url", "directions_url",
        "reservation_url", "collected_at",
    ]
    df = pd.DataFrame(rows, columns=columns)

    for numeric_column in ["capacity_personal", "capacity_group", "age_min", "age_max"]:
        if numeric_column in df.columns:
            df[numeric_column] = pd.array(df[numeric_column], dtype="Int64")

    return df


def build_review_docs_dataframe(docs: list[ReviewDoc], *, collected_at: str) -> pd.DataFrame:
    # review_docs 테이블은 후기성 문서 1행 기준 테이블입니다.
    rows: list[dict[str, Any]] = []
    for index, doc in enumerate(docs, start=1):
        rows.append(
            {
                "doc_id": f"review_{index:04d}",
                "place_id": doc.place_id,
                "source_type": doc.source_type,
                "title": doc.title,
                "content": doc.content,
                "doc_url": doc.doc_url,
                "post_date": normalize_post_date_value(doc.post_date),
                "author_name": doc.author_name,
            }
        )

    columns = [
        "doc_id", "place_id", "source_type", "title", "content", "doc_url",
        "post_date", "author_name",
    ]
    return pd.DataFrame(rows, columns=columns)


def build_place_features_dataframe(
    places: list[PortalPlace],
    docs: list[ReviewDoc],
    *,
    collected_at: str,
) -> pd.DataFrame:
    # place_features 테이블은 매장 1행 기준의 가로형 feature 테이블입니다.
    rows: list[dict[str, Any]] = []

    for place in places:
        rows.extend(extract_place_feature_rows(place))

    if INCLUDE_REVIEW_DERIVED_FEATURES:
        rows.extend(build_review_feature_rows(docs))

    columns = ["place_id", *SELECTED_FEATURE_NAMES]
    if not rows:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(rows)
    df = df[df["feature_name"].isin(SELECTED_FEATURE_NAMES)].copy()
    if df.empty:
        return pd.DataFrame(columns=columns)

    # 같은 매장/feature에 여러 후보가 있으면 confidence가 가장 높은 값을 채택합니다.
    df = (
        df.sort_values(["place_id", "feature_name", "confidence"], ascending=[True, True, False])
          .drop_duplicates(subset=["place_id", "feature_name"], keep="first")
          .copy()
    )

    wide_df = df.pivot(index="place_id", columns="feature_name", values="feature_value").reset_index()
    wide_df.columns.name = None

    for feature_name in SELECTED_FEATURE_NAMES:
        if feature_name not in wide_df.columns:
            wide_df[feature_name] = ""

    wide_df = wide_df[["place_id", *SELECTED_FEATURE_NAMES]].fillna("")
    return wide_df


def build_output_tables(places: list[PortalPlace], docs: list[ReviewDoc]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # 추천 챗봇에서 바로 쓰기 좋도록 3개 테이블 형태로 나눕니다.
    collected_at = datetime.now().astimezone().isoformat(timespec="seconds")
    places_df = build_places_dataframe(places, collected_at=collected_at)
    review_docs_df = build_review_docs_dataframe(docs, collected_at=collected_at)
    place_features_df = build_place_features_dataframe(places, docs, collected_at=collected_at)
    return places_df, review_docs_df, place_features_df


In [ ]:
# -----------------------------
# 설정 확인
# -----------------------------
# 위쪽 `사용자 설정` 셀에서 바꾼 값이 아래처럼 반영되는지 확인합니다.

settings_summary = {
    'use_env_file': USE_ENV_FILE,
    'env_file_name': ENV_FILE_NAME,
    'naver_client_id_set': bool(NAVER_CLIENT_ID),
    'naver_client_secret_set': bool(NAVER_CLIENT_SECRET),
    'facility_style_code': FACILITY_STYLE_CODE,
    'district_count': len(SELECTED_DISTRICT_CODES),
    'max_pages': MAX_PAGES,
    'enrich_detail': ENRICH_DETAIL,
    'enrich_directions': ENRICH_DIRECTIONS,
    'enrich_program': ENRICH_PROGRAM,
    'include_blog': INCLUDE_BLOG,
    'blog_docs_per_place': BLOG_DOCS_PER_PLACE,
    'include_cafe': INCLUDE_CAFE,
    'cafe_docs_per_place': CAFE_DOCS_PER_PLACE,
    'review_terms': REVIEW_TERMS,
    'doc_sort': DOC_SORT,
    'filter_promotional_docs': FILTER_PROMOTIONAL_DOCS,
    'promotional_keyword_count': len(PROMOTIONAL_KEYWORDS),
    'include_review_derived_features': INCLUDE_REVIEW_DERIVED_FEATURES,
    'request_interval_seconds': REQUEST_INTERVAL_SECONDS,
    'request_max_retries': REQUEST_MAX_RETRIES,
    'output_dir': 'outputs/raw',
    'output_files': ['places.csv', 'review_docs.csv', 'place_features.csv'],
}

pd.DataFrame(settings_summary.items(), columns=['setting', 'value'])


In [ ]:
# -----------------------------
# 1) 서울형 키즈카페 포털 메타데이터 수집
# -----------------------------

places = collect_portal_places(
    facility_style_code=FACILITY_STYLE_CODE,
    district_codes=SELECTED_DISTRICT_CODES,
    max_pages=MAX_PAGES,
    enrich_detail=ENRICH_DETAIL,
    enrich_directions=ENRICH_DIRECTIONS,
    enrich_program=ENRICH_PROGRAM,
)

raw_places_df = portal_places_to_dataframe(places)
print(f"portal_places={len(raw_places_df)}")
display(raw_places_df.head())


In [ ]:
# -----------------------------
# 1-1) places 테이블 형태 미리보기
# -----------------------------
# places는 추천 필터링의 기준 테이블입니다.

places_preview_df = build_places_dataframe(
    places,
    collected_at=datetime.now().astimezone().isoformat(timespec="seconds"),
)
print(f"places_table_rows={len(places_preview_df)}")
display(places_preview_df.head())


In [ ]:
# -----------------------------
# 2) 네이버 블로그/카페글 검색 결과 수집
# -----------------------------
# 이 단계는 NAVER_CLIENT_ID / NAVER_CLIENT_SECRET이 준비돼 있어야 합니다.
# .env.local에 아래 두 줄이 있으면 자동으로 읽습니다.
#
# export NAVER_CLIENT_ID=...
# export NAVER_CLIENT_SECRET=...

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise RuntimeError("NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수를 먼저 설정해주세요.")

review_docs: list[ReviewDoc] = []

if INCLUDE_BLOG and BLOG_DOCS_PER_PLACE > 0:
    review_docs.extend(
        search_review_docs(
            places=places,
            endpoint="blog",
            source_type="naver_blog",
            client_id=NAVER_CLIENT_ID,
            client_secret=NAVER_CLIENT_SECRET,
            docs_per_place=BLOG_DOCS_PER_PLACE,
            review_terms=REVIEW_TERMS,
            sort=DOC_SORT,
        )
    )

if INCLUDE_CAFE and CAFE_DOCS_PER_PLACE > 0:
    review_docs.extend(
        search_review_docs(
            places=places,
            endpoint="cafearticle",
            source_type="naver_cafe",
            client_id=NAVER_CLIENT_ID,
            client_secret=NAVER_CLIENT_SECRET,
            docs_per_place=CAFE_DOCS_PER_PLACE,
            review_terms=REVIEW_TERMS,
            sort=DOC_SORT,
        )
    )

print(f"review_docs={len(review_docs)}")
pd.DataFrame([asdict(doc) for doc in review_docs]).head()


In [ ]:
# -----------------------------
# 3) 구조화 테이블 생성
# -----------------------------
# 추천 시스템에서 바로 쓰기 좋도록 places / review_docs / place_features로 나눕니다.

places_df, review_docs_df, place_features_df = build_output_tables(places, review_docs)

summary_df = pd.DataFrame(
    [
        {"table_name": "places", "rows": len(places_df), "columns": len(places_df.columns)},
        {"table_name": "review_docs", "rows": len(review_docs_df), "columns": len(review_docs_df.columns)},
        {"table_name": "place_features", "rows": len(place_features_df), "columns": len(place_features_df.columns)},
    ]
)
display(summary_df)

display(places_df.head())
display(review_docs_df.head())
display(place_features_df.head())


In [ ]:
# -----------------------------
# 4) 구조화 CSV 저장
# -----------------------------
# 파일명은 항상 단순하게 유지합니다. 다시 실행하면 같은 이름으로 덮어씁니다.

places_output_path = RAW_OUTPUT_DIR / "places.csv"
review_docs_output_path = RAW_OUTPUT_DIR / "review_docs.csv"
place_features_output_path = RAW_OUTPUT_DIR / "place_features.csv"

places_df.to_csv(places_output_path, index=False, encoding="utf-8-sig")
review_docs_df.to_csv(review_docs_output_path, index=False, encoding="utf-8-sig")
place_features_df.to_csv(place_features_output_path, index=False, encoding="utf-8-sig")

display(
    pd.DataFrame(
        [
            {"table_name": "places", "output_path": str(places_output_path)},
            {"table_name": "review_docs", "output_path": str(review_docs_output_path)},
            {"table_name": "place_features", "output_path": str(place_features_output_path)},
        ]
    )
)
